## Analiza rozmów sprzedażowych: obiekcje i zbijanie obiekcji (prototyp)- Pytanie biznesowe: Jak doradcy radzą sobie z obiekcjami klientów?- Zakres: identyfikacja rozmów sprzedażowych, detekcja obiekcji, kategoryzacja reakcji doradców, raport.- Założenia: minimalizacja zapytań do LLM; połączenie reguł, embeddingów i prostych modeli.- Dane: `transcripts_combined - sample2.csv`

In [ ]:
import osimport reimport randomfrom typing import Listimport pandas as pdimport numpy as npfrom sentence_transformers import SentenceTransformer, utilimport matplotlib.pyplot as pltimport seaborn as snsSEED = 42random.seed(SEED)np.random.seed(SEED)DATA_PATH = "/workspace/transcripts_combined - sample2.csv"SAMPLE_SIZE = NoneEMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"print("DATA_PATH:", DATA_PATH)model = SentenceTransformer(EMBED_MODEL)

In [ ]:
usecols = ["filename", "full_text", "confidence", "audio_duration_seconds", "word_count", "redacted_pii_policies"]df = pd.read_csv(DATA_PATH, usecols=usecols)if SAMPLE_SIZE:    df = df.sample(min(SAMPLE_SIZE, len(df)), random_state=SEED).reset_index(drop=True)print(df.shape)df.head(2)

In [ ]:
# Heurystyki rozmów sprzedażowych i segmentacjaSALES_CUES = [    r\bmedicare\b, r\bbenefit, r\bno additional cost\b, r\bqualified\b,    r\btransfer(ing)? your call\b, r\b[OCCUPATION]\b, r\b[ORGANIZATION]\b,    r\boffer\b, r\bplan\b, r\bcoverage\b]def is_sales(text: str) -> bool:    t = str(text).lower()    return any(re.search(p, t) for p in SALES_CUES)df["is_sales"] = df["full_text"].apply(is_sales)sales_df = df[df["is_sales"]].copy().reset_index(drop=True)sent_split = re.compile(r"(?<=[.!?])\s+")def split_sentences(text: str) -> List[str]:    text = str(text).replace("\n", " ")    return [s.strip() for s in sent_split.split(text) if len(s.strip()) > 1]AGENT_CUES = ["my name is", "this is", "thank you for holding", "transfer", "benefit", "zip code", "verify", "date of birth", "[organization]", "[occupation]", "no additional cost"]CUSTOMER_CUES = ["not interested", "don't have time", "no time", "call me back", "who are you", "what is this", "i already have", "i don't", "i can't", "not now", "busy", "have to leave", "i'm in a rush", "i think", "i guess", "i'm not sure", "i didn't understand"]def tag_speaker(sentence: str) -> str:    s = sentence.lower()    a = sum(c in s for c in AGENT_CUES)    c = sum(c in s for c in CUSTOMER_CUES)    if a == 0 and c == 0:        if any(k in s for k in ["medicare", "benefit", "transfer", "zip code", "verify", "date of birth"]):            return "agent"        return "unknown"    return "agent" if a >= c else "customer"sales_df["sentences"] = sales_df["full_text"].apply(split_sentences)sales_df["speaker_tags"] = sales_df["sentences"].apply(lambda sents: [tag_speaker(s) for s in sents])len(sales_df)

In [ ]:
# Detekcja obiekcji po stronie klienta (reguły)OBJECTION_PATTERNS = [    r\bnot interested\b, r\bdon't have time\b|\bno time\b|\bbusy\b|\bin a rush\b,    r\btoo expensive\b|\btoo much\b|\bcan't afford\b|\bthat's a lot\b,    r\bi already have\b|\bdon't need\b|\bdon't want\b,    r\bcall me back\b|\bnot now\b,    r\bi'm not sure\b|\bi didn't (really )?understand\b|\bwhat is this\b|\bwho are you\b,]OBJECTION_TYPES = ["no_interest", "no_time", "price", "no_need", "delay", "confusion"]compiled_patterns = [re.compile(p) for p in OBJECTION_PATTERNS]records = []for idx, row in sales_df.iterrows():    for i, sent in enumerate(row["sentences"]):        if row["speaker_tags"][i] != "customer":            continue        s_low = sent.lower()        matches = [j for j, pat in enumerate(compiled_patterns) if pat.search(s_low)]        if matches:            records.append({                "call_id": idx,                "filename": row["filename"],                "sent_idx": i,                "sentence": sent,                "objection_types": [OBJECTION_TYPES[j] for j in matches]            })objections_df = pd.DataFrame(records)print(objections_df.shape)objections_df.head(3)

In [ ]:
# Kategoryzacja reakcji doradców (embeddingi)TACTIC_TEMPLATES = {    "value_reframing": [        "these benefits will save you money / reduce co-pay / no additional cost",        "you will receive dental, vision, hearing and transportation benefits",        "this will be added to your existing plan at no cost"    ],    "social_proof_assurance": [        "we work with [organization], calls are recorded for quality",        "advisor from your area/zip code will explain details",        "many clients in your area benefit from this"    ],    "clarification": [        "let me explain how it works / what this call is about",        "i'll clarify your eligibility and steps",        "this is additional benefits, not changing your plan"    ],    "alternative_next_step": [        "i will connect your call to an advisor now",        "we can schedule a callback",        "let me send paperwork so you can read"    ],    "urgency_scarcity": [        "updated for this year only",        "eligible today / now",        "limited time to connect"    ]}all_templates = [(tactic, phrase) for tactic, phrases in TACTIC_TEMPLATES.items() for phrase in phrases]template_texts = [p for _, p in all_templates]template_emb = model.encode(template_texts, convert_to_tensor=True, normalize_embeddings=True)def categorize_response(agent_sentences: List[str]):    if not agent_sentences:        return ("unknown", 0.0, "")    cand_text = " ".join(agent_sentences[:3])[:500]    cand_emb = model.encode([cand_text], convert_to_tensor=True, normalize_embeddings=True)    sim = util.cos_sim(cand_emb, template_emb).cpu().numpy()[0]    best_idx = int(np.argmax(sim))    best_score = float(sim[best_idx])    tactic = all_templates[best_idx][0]    return tactic, best_score, cand_textresponse_rows = []WINDOW = 5for _, evt in objections_df.iterrows():    call = sales_df.iloc[evt["call_id"]]    start = evt["sent_idx"] + 1    agent_follow = []    for s, tag in zip(call["sentences"][start:start+15], call["speaker_tags"][start:start+15]):        if len(agent_follow) >= WINDOW:            break        if tag == "agent":            agent_follow.append(s)        elif agent_follow:            break    tactic, score, preview = categorize_response(agent_follow)    response_rows.append({        "filename": evt["filename"],        "sent_idx": evt["sent_idx"],        "objection_types": evt["objection_types"],        "agent_resp_tactic": tactic,        "agent_resp_conf": score,        "agent_resp_preview": preview    })responses_df = pd.DataFrame(response_rows)responses_df.head(5)

In [ ]:
# Raporty i eksportfrom collections import Countertactic_counts = responses_df["agent_resp_tactic"].value_counts()obj_flat = [t for lst in objections_df["objection_types"] for t in lst]obj_counts = pd.Series(Counter(obj_flat)).sort_values(ascending=False)rows = []for _, r in responses_df.iterrows():    for o in r["objection_types"]:        rows.append((o, r["agent_resp_tactic"]))ct = pd.crosstab(pd.Series([r[0] for r in rows], name="objection"), pd.Series([r[1] for r in rows], name="tactic"))os.makedirs("/workspace/reports", exist_ok=True)responses_df.to_csv("/workspace/reports/objection_events_with_tactics.csv", index=False)ct.to_csv("/workspace/reports/objection_tactic_crosstab.csv")obj_counts.to_csv("/workspace/reports/objection_type_counts.csv")tactic_counts.to_csv("/workspace/reports/tactic_counts.csv")print("Saved reports to /workspace/reports")

### Ewaluacja i porównywalność modeli- Metryki: Precision/Recall/F1 dla: is_sales, is_objection, tactic.- Adnotacja: próbka losowa zdarzeń (CSV) z kolumnami `gold_is_objection`, `gold_tactic`.- Porównanie: funkcja `categorize_response()` jako wymienialny moduł.

In [ ]:
# Próbka do adnotacjiSAMPLE_ANN_SIZE = 100sample_ann = responses_df.sample(min(SAMPLE_ANN_SIZE, len(responses_df)), random_state=SEED).copy()sample_ann["gold_is_objection"] = ""sample_ann["gold_tactic"] = ""sample_ann.to_csv("/workspace/reports/annotation_sample.csv", index=False)print("Created /workspace/reports/annotation_sample.csv")